In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/audiotext/test_sent_emo.csv
/kaggle/input/audiotext/text_emotion.pkl
/kaggle/input/audiotext/audio_emotion.pkl
/kaggle/input/audiotext/dev_sent_emo.csv
/kaggle/input/audiotext/train_sent_emo.csv
/kaggle/input/wesad-wearable-stress-affect-detection-dataset/WESAD/wesad_readme.pdf
/kaggle/input/wesad-wearable-stress-affect-detection-dataset/WESAD/S14/S14.pkl
/kaggle/input/wesad-wearable-stress-affect-detection-dataset/WESAD/S14/S14_quest.csv
/kaggle/input/wesad-wearable-stress-affect-detection-dataset/WESAD/S14/S14_respiban.txt
/kaggle/input/wesad-wearable-stress-affect-detection-dataset/WESAD/S14/S14_readme.txt
/kaggle/input/wesad-wearable-stress-affect-detection-dataset/WESAD/S14/S14_E4_Data/HR.csv
/kaggle/input/wesad-wearable-stress-affect-detection-dataset/WESAD/S14/S14_E4_Data/IBI.csv
/kaggle/input/wesad-wearable-stress-affect-detection-dataset/WESAD/S14/S14_E4_Data/BVP.csv
/kaggle/input/wesad-wearable-stress-affect-detection-dataset/WESAD/S14/S14_E4_Data/EDA.csv
/kaggl

In [4]:
# ─── 0️⃣ Install & Imports ────────────────────────────────────────────────
!pip install --quiet scipy scikit-learn tqdm

import os, glob, pickle
import numpy as np
import pandas as pd
from collections import Counter
from scipy.signal            import butter, filtfilt, resample_poly, find_peaks
from sklearn.ensemble        import RandomForestClassifier
from sklearn.linear_model   import LogisticRegression
from sklearn.preprocessing  import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics        import classification_report, accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')

# ─── 1️⃣ Paths ─────────────────────────────────────────────────────────────
WESAD_DIR     = "/kaggle/input/wesad-wearable-stress-affect-detection-dataset/WESAD"
AUDIOTEXT_DIR = "/kaggle/input/audiotext"

# ─── 2️⃣ WESAD Physiological Pipeline ──────────────────────────────────────
def convert_units(sig, mod):
    if mod=='EDA':    return ((sig/65536.0)*3.0)/0.12
    if mod=='ECG':    return ((sig/65536.0)-0.5)*3000.0
    if mod=='RESP':   return ((sig/65536.0)-0.5)*100.0
    return sig

def lowpass(sig, cutoff, fs):
    nyq = fs*0.5
    if cutoff>=nyq or sig.size<10:
        return sig
    b,a = butter(4, cutoff/nyq, btype='low')
    try:
        return filtfilt(b,a,sig)
    except:
        return sig

def downsample(sig, frm, to, mod=None):
    if mod: sig = convert_units(sig, mod)
    if frm==to: return sig
    f = lowpass(sig, to*0.5, frm)
    return resample_poly(f, to, frm)

def basic_stats(x):
    x = x.flatten()
    return [
        x.mean(), x.std(), x.min(), x.max(),
        np.percentile(x,25), np.percentile(x,50), np.percentile(x,75),
        x.ptp(), np.mean(np.abs(np.diff(x)))
    ]

def ecg_feats(x, fs):
    x = x.flatten()
    peaks,_ = find_peaks(x, distance=fs*0.5)
    rr = np.diff(peaks)/fs if len(peaks)>1 else np.array([])
    hr  = 60/rr.mean() if rr.size>0 else 0
    hrv = rr.std()    if rr.size>0 else 0
    return [hr, hrv] + basic_stats(x)

def build_physio_features(root_dir=WESAD_DIR,
                          fs_orig=700, fs_ecg=64, fs_other=4,
                          win_s=4, stride_s=2):
    WIN    = fs_orig*win_s
    STRIDE = fs_orig*stride_s
    Xp, yp = [], []
    pkl_paths = glob.glob(os.path.join(root_dir,"S*","S*.pkl"))
    print(f"▶️ Found {len(pkl_paths)} WESAD subjects")
    for p in pkl_paths:
        with open(p,'rb') as f:
            data = pickle.load(f, encoding='latin1')
        ecg, eda, resp = data['signal']['chest']['ECG'], data['signal']['chest']['EDA'], data['signal']['chest']['Resp']
        labels = data['label']  # 700Hz

        # downsample once per subject
        ecg_ds  = downsample(ecg,  fs_orig, fs_ecg,  'ECG')
        eda_ds  = downsample(eda,  fs_orig, fs_other,'EDA')
        resp_ds = downsample(resp, fs_orig, fs_other,'RESP')

        # slide window on raw labels
        for start in range(0, len(labels)-WIN+1, STRIDE):
            lbl, _ = Counter(labels[start:start+WIN]).most_common(1)[0]
            if lbl not in (1,2,3,4):
                continue
            e0,e1 = int(start/fs_orig*fs_ecg), int((start+WIN-1)/fs_orig*fs_ecg)+1
            o0,o1 = int(start/fs_orig*fs_other), int((start+WIN-1)/fs_orig*fs_other)+1
            feats = (
                ecg_feats(ecg_ds[e0:e1], fs_ecg) +
                basic_stats(eda_ds[o0:o1]) +
                basic_stats(resp_ds[o0:o1])
            )
            Xp.append(feats)
            yp.append(lbl-1)  # zero-based [0…3]

    return np.array(Xp), np.array(yp)

# build & split
X_physio, y_physio = build_physio_features()
Xp_tr, Xp_te, yp_tr, yp_te = train_test_split(
    X_physio, y_physio, test_size=0.3, stratify=y_physio, random_state=42
)

# train RF on WESAD
rf_physio = RandomForestClassifier(
    n_estimators=200, class_weight='balanced', random_state=42
).fit(Xp_tr, yp_tr)

# quick eval
print("\n=== Physio-Only RF on WESAD Test ===")
print(classification_report(yp_te, rf_physio.predict(Xp_te),
                            target_names=['baseline','stress','amusement','meditation']))

physio_probs_tr = rf_physio.predict_proba(Xp_tr)
physio_probs_te = rf_physio.predict_proba(Xp_te)


# ─── 3️⃣ MELD-Style Text & Audio Pipeline ─────────────────────────────────
# 3.1 load CSV splits
train_df = pd.read_csv( os.path.join(AUDIOTEXT_DIR,"train_sent_emo.csv") )
dev_df   = pd.read_csv( os.path.join(AUDIOTEXT_DIR,"dev_sent_emo.csv")   )
test_df  = pd.read_csv( os.path.join(AUDIOTEXT_DIR,"test_sent_emo.csv")  )
print(f"\n▶️ Loaded MELD splits: train={len(train_df)}, dev={len(dev_df)}, test={len(test_df)}")

# 3.2 build global label array
all_df = pd.concat([train_df, dev_df, test_df], ignore_index=True)
if 'Emotion' in all_df.columns:
    # use BaseMELD mapping:
    emo_map = {'neutral':0,'surprise':1,'fear':2,'sadness':3,'joy':4,'disgust':5,'anger':6}
    all_df['y'] = all_df['Emotion'].map(emo_map).fillna(0).astype(int)
else:
    # already has 'label'
    all_df['y'] = all_df['label']
y_all = all_df['y'].values
n1,n2 = len(train_df), len(dev_df)

# 3.3 load the raw feature pickles
with open(os.path.join(AUDIOTEXT_DIR,"text_emotion.pkl"),'rb')  as f: text_data  = pickle.load(f)
with open(os.path.join(AUDIOTEXT_DIR,"audio_emotion.pkl"),'rb') as f: audio_data = pickle.load(f)

# 3.4 helper to flatten MELD pickles (list-of-3-dicts) → (N_utt, feat_dim)
def extract_meld_feats(data, total):
    feats = []
    for split in data:
        for _, arr in split.items():
            feats.append(arr)
    feats = np.vstack(feats)
    if feats.shape[0] > total:
        feats = feats[:total]
    elif feats.shape[0] < total:
        pad = total - feats.shape[0]
        feats = np.vstack([feats, np.zeros((pad, feats.shape[1]))])
    return feats

X_text_all  = extract_meld_feats(text_data,  len(y_all))
X_audio_all = extract_meld_feats(audio_data, len(y_all))

# split back
X_text_tr, X_text_dev, X_text_te   = np.split(X_text_all , [n1, n1+n2])
X_audio_tr, X_audio_dev, X_audio_te= np.split(X_audio_all, [n1, n1+n2])
y_tr = y_all[:n1]
y_dev= y_all[n1:n1+n2]
y_te = y_all[n1+n2:]

# 3.5 scale
sc_txt = StandardScaler().fit(np.vstack([X_text_tr, X_text_dev]))
X_text_tr = sc_txt.transform(X_text_tr)
X_text_dev= sc_txt.transform(X_text_dev)
X_text_te = sc_txt.transform(X_text_te)

sc_aud = StandardScaler().fit(np.vstack([X_audio_tr, X_audio_dev]))
X_audio_tr = sc_aud.transform(X_audio_tr)
X_audio_dev= sc_aud.transform(X_audio_dev)
X_audio_te = sc_aud.transform(X_audio_te)

# 3.6 train base LRs
lr_txt = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_aud = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)

lr_txt.fit(np.vstack([X_text_tr,X_text_dev]), np.hstack([y_tr,y_dev]))
lr_aud.fit(np.vstack([X_audio_tr,X_audio_dev]), np.hstack([y_tr,y_dev]))

print("\n=== Text-Only LR on MELD Test ===")
print(classification_report(y_te, lr_txt.predict(X_text_te)))
print("\n=== Audio-Only LR on MELD Test ===")
print(classification_report(y_te, lr_aud.predict(X_audio_te)))

text_probs_tr  = lr_txt .predict_proba(np.vstack([X_text_tr,X_text_dev]))
text_probs_te  = lr_txt .predict_proba(X_text_te)
audio_probs_tr = lr_aud.predict_proba(np.vstack([X_audio_tr,X_audio_dev]))
audio_probs_te = lr_aud.predict_proba(X_audio_te)


# ─── 4️⃣ FUSION OF ALL THREE MODALITIES ────────────────────────────────────
# align physio train → we’ll just take the first N_train samples
physio_probs_train = physio_probs_tr[: text_probs_tr.shape[0] ]

# (a) Late-fusion average
avg_probs = ( text_probs_te + audio_probs_te + physio_probs_te ) / 3
pred_avg  = avg_probs.argmax(axis=1)
print("\n--- Late-Fusion Avg on MELD Test ---")
print(classification_report(y_te, pred_avg))

# (b) Soft Voting
print("\n--- Soft Voting on MELD Test ---")
print(classification_report(y_te, pred_avg))

# (c) Hard Voting
hard = []
for t,a,p in zip(text_probs_te.argmax(1), audio_probs_te.argmax(1), physio_probs_te.argmax(1)):
    hard.append(Counter([t,a,p]).most_common(1)[0][0])
print("\n--- Hard Voting on MELD Test ---")
print(classification_report(y_te, hard))

# (d) Stacking (Logistic Regression meta-learner)
X_meta_tr = np.hstack([ text_probs_tr,  audio_probs_tr,  physio_probs_train ])
X_meta_te = np.hstack([ text_probs_te,  audio_probs_te,  physio_probs_te    ])

meta = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
meta.fit(X_meta_tr, np.hstack([y_tr,y_dev]))
pred_stack = meta.predict(X_meta_te)

print("\n--- Stacking Ensemble on MELD Test ---")
print(classification_report(y_te, pred_stack))


▶️ Found 15 WESAD subjects

=== Physio-Only RF on WESAD Test ===
              precision    recall  f1-score   support

    baseline       0.97      0.97      0.97      2641
      stress       0.97      0.98      0.97      1494
   amusement       0.94      0.89      0.92       837
  meditation       0.95      0.97      0.96      1772

    accuracy                           0.96      6744
   macro avg       0.96      0.95      0.95      6744
weighted avg       0.96      0.96      0.96      6744


▶️ Loaded MELD splits: train=2610, dev=1109, test=2610

=== Text-Only LR on MELD Test ===
              precision    recall  f1-score   support

           0       0.42      0.13      0.19      1256
           1       0.12      0.10      0.11       281
           2       0.00      0.02      0.01        50
           3       0.08      0.16      0.11       208
           4       0.15      0.18      0.17       402
           5       0.03      0.18      0.06        68
           6       0.12      0

ValueError: operands could not be broadcast together with shapes (2610,7) (6744,4) 

In [6]:
# ─── 0️⃣ INSTALL & IMPORTS ────────────────────────────────────────────────
!pip install --quiet scipy scikit-learn tqdm

import os, glob, pickle
import numpy as np
import pandas as pd
from collections import Counter
from scipy.signal            import butter, filtfilt, resample_poly, find_peaks
from sklearn.ensemble        import RandomForestClassifier
from sklearn.linear_model   import LogisticRegression
from sklearn.preprocessing  import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics        import classification_report, accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')

# ─── 1️⃣ BUILD & TRAIN WESAD PHYSIO MODEL ────────────────────────────────
def convert_units(sig, mod):
    if mod=='EDA':    return ((sig/65536.0)*3.0)/0.12
    if mod=='ECG':    return ((sig/65536.0)-0.5)*3000.0
    if mod=='RESP':   return ((sig/65536.0)-0.5)*100.0
    return sig

def lowpass(sig, cutoff, fs):
    nyq = fs*0.5
    if cutoff >= nyq or sig.size < 10:
        return sig
    b,a = butter(4, cutoff/nyq, btype='low')
    try:
        return filtfilt(b,a,sig)
    except:
        return sig

def downsample(sig, frm, to, mod=None):
    if mod:
        sig = convert_units(sig, mod)
    if frm == to:
        return sig
    f = lowpass(sig, to*0.5, frm)
    return resample_poly(f, to, frm)

def basic_stats(x):
    x = x.flatten()
    return [
        x.mean(), x.std(), x.min(), x.max(),
        np.percentile(x,25), np.percentile(x,50), np.percentile(x,75),
        x.ptp(), np.mean(np.abs(np.diff(x)))
    ]

def ecg_feats(x, fs):
    x = x.flatten()
    peaks,_ = find_peaks(x, distance=fs*0.5)
    rr = np.diff(peaks)/fs if len(peaks)>1 else np.array([])
    hr  = 60/rr.mean() if rr.size>0 else 0
    hrv = rr.std()    if rr.size>0 else 0
    return [hr, hrv] + basic_stats(x)

def build_physio(root="/kaggle/input/wesad-wearable-stress-affect-detection-dataset/WESAD"):
    fs_orig, fs_ecg, fs_other = 700, 64, 4
    win, stride = fs_orig*4, fs_orig*2
    Xp, yp = [], []
    for pkl_fp in glob.glob(os.path.join(root, "S*", "S*.pkl")):
        with open(pkl_fp,'rb') as f:
            data = pickle.load(f, encoding='latin1')
        ecg, eda, resp = data['signal']['chest']['ECG'], data['signal']['chest']['EDA'], data['signal']['chest']['Resp']
        lbl700 = data['label']
        ecg_ds  = downsample(ecg,  fs_orig, fs_ecg,  'ECG')
        eda_ds  = downsample(eda,  fs_orig, fs_other,'EDA')
        resp_ds = downsample(resp, fs_orig, fs_other,'RESP')
        for i in range(0, len(lbl700)-win+1, stride):
            lab,_ = Counter(lbl700[i:i+win]).most_common(1)[0]
            if lab not in (1,2,3,4): 
                continue
            e0,e1 = int(i/fs_orig*fs_ecg), int((i+win-1)/fs_orig*fs_ecg)+1
            o0,o1 = int(i/fs_orig*fs_other), int((i+win-1)/fs_orig*fs_other)+1
            feats = (
                ecg_feats(ecg_ds[e0:e1], fs_ecg)
                + basic_stats(eda_ds[o0:o1])
                + basic_stats(resp_ds[o0:o1])
            )
            Xp.append(feats)
            yp.append(lab-1)  # zero-based
    return np.array(Xp), np.array(yp)

X_physio, y_physio = build_physio()
Xp_tr, Xp_te, yp_tr, yp_te = train_test_split(
    X_physio, y_physio, test_size=0.3, stratify=y_physio, random_state=42
)

rf_physio = RandomForestClassifier(
    n_estimators=200, class_weight='balanced', random_state=42
).fit(Xp_tr, yp_tr)

print("\n=== Physio-Only RF on WESAD Test ===")
print(classification_report(yp_te, rf_physio.predict(Xp_te),
      target_names=['baseline','stress','amusement','meditation']))

# save physio probs
physio_probs_tr = rf_physio.predict_proba(Xp_tr)
physio_probs_te = rf_physio.predict_proba(Xp_te)


# ─── 2️⃣ LOAD & TRAIN MELD TEXT/AUDIO MODELS ───────────────────────────────
AUDIOTEXT = "/kaggle/input/audiotext"
train_df = pd.read_csv(os.path.join(AUDIOTEXT,"train_sent_emo.csv"))
dev_df   = pd.read_csv(os.path.join(AUDIOTEXT,"dev_sent_emo.csv"))
test_df  = pd.read_csv(os.path.join(AUDIOTEXT,"test_sent_emo.csv"))

N1, N2, N3 = len(train_df), len(dev_df), len(test_df)
print(f"\n▶️ MELD splits: train={N1}, dev={N2}, test={N3}")

# assemble y arrays
if 'label' in train_df.columns:
    y_train = train_df['label'].values
    y_dev   = dev_df  ['label'].values
    y_test  = test_df ['label'].values
else:
    # fallback if using 'Emotion' strings
    emo_map = {'neutral':0,'surprise':1,'fear':2,'sadness':3,'joy':4,'disgust':5,'anger':6}
    y_train = train_df['Emotion'].map(emo_map).fillna(0).astype(int).values
    y_dev   = dev_df  ['Emotion'].map(emo_map).fillna(0).astype(int).values
    y_test  = test_df ['Emotion'].map(emo_map).fillna(0).astype(int).values

# load pickles
with open(os.path.join(AUDIOTEXT,"text_emotion.pkl"), 'rb')  as f: text_data  = pickle.load(f)
with open(os.path.join(AUDIOTEXT,"audio_emotion.pkl"),'rb') as f: audio_data = pickle.load(f)

def extract_meld(data, total):
    feats=[]
    for split in data:
        for _,arr in split.items():
            feats.append(arr)
    feats = np.vstack(feats)
    if feats.shape[0]>total: feats=feats[:total]
    if feats.shape[0]<total:
        pad=total-feats.shape[0]
        feats=np.vstack([feats, np.zeros((pad,feats.shape[1]))])
    return feats

# get full features
X_text_all  = extract_meld(text_data,  N1+N2+N3)
X_audio_all = extract_meld(audio_data, N1+N2+N3)

# split
X_text_tr, X_text_dev, X_text_te   = np.split(X_text_all,  [N1, N1+N2])
X_audio_tr, X_audio_dev, X_audio_te= np.split(X_audio_all, [N1, N1+N2])

# scale
sc_t = StandardScaler().fit(np.vstack([X_text_tr,X_text_dev]))
X_text_tr  = sc_t.transform(X_text_tr)
X_text_dev = sc_t.transform(X_text_dev)
X_text_te  = sc_t.transform(X_text_te)

sc_a = StandardScaler().fit(np.vstack([X_audio_tr,X_audio_dev]))
X_audio_tr  = sc_a.transform(X_audio_tr)
X_audio_dev = sc_a.transform(X_audio_dev)
X_audio_te  = sc_a.transform(X_audio_te)

# train 7-way LRs on train+dev
lr_txt = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_aud = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)

lr_txt.fit(np.vstack([X_text_tr,X_text_dev]), np.hstack([y_train,y_dev]))
lr_aud.fit(np.vstack([X_audio_tr,X_audio_dev]),np.hstack([y_train,y_dev]))

print("\n=== Text-only LR on MELD Test ===")
print(classification_report(y_test, lr_txt.predict(X_text_te)))
print("\n=== Audio-only LR on MELD Test ===")
print(classification_report(y_test, lr_aud.predict(X_audio_te)))

# get dev/test probs
text_probs_dev  = lr_txt .predict_proba(X_text_dev)
audio_probs_dev = lr_aud.predict_proba(X_audio_dev)
text_probs_te   = lr_txt .predict_proba(X_text_te)
audio_probs_te  = lr_aud.predict_proba(X_audio_te)

# slice physio to utterance-level counts
#   we simply take the first N1 windows from physio_probs_tr as "train utterances",
#   next N2 windows as "dev utterances", and first N3 of physio_probs_te as test.
pp_tr = physio_probs_tr[:N1, 1]             # physio stress-prob on train
pp_dev= physio_probs_tr[N1:N1+N2, 1]        # physio stress-prob on dev
pp_te = physio_probs_te[:N3, 1]             # physio stress-prob on test

# ─── 3️⃣ BINARY STRESS VS NON-STRESS FUSION ──────────────────────────────
# map MELD 7-way → stress vs non-stress (fear,sadness,disgust,anger → stress)
stress_idxs = [2,3,5,6]
y_dev_bin = np.isin(y_dev, stress_idxs).astype(int)
y_te_bin  = np.isin(y_test, stress_idxs).astype(int)

# collapse to scalar stress probs
tp_dev = text_probs_dev[:, stress_idxs].sum(axis=1)
ap_dev = audio_probs_dev[:, stress_idxs].sum(axis=1)

tp_te  = text_probs_te[:, stress_idxs].sum(axis=1)
ap_te  = audio_probs_te[:, stress_idxs].sum(axis=1)

# ─── 4️⃣ GRID-SEARCH FUSION WEIGHTS on DEV ─────────────────────────────────
best = (0,0,0,-1)
grid = np.linspace(0,1,21)
for wt in grid:
    for wa in grid:
        wp = 1 - wt - wa
        if wp<0 or wp>1: continue
        fused_dev = wt*tp_dev + wa*ap_dev + wp*pp_dev
        preds_dev = (fused_dev>0.5).astype(int)
        f1 = f1_score(y_dev_bin, preds_dev)
        if f1>best[3]:
            best = (wt,wa,wp,f1)

wt, wa, wp, best_f1 = best
print(f"\n→ Best dev weights: text={wt:.2f}, audio={wa:.2f}, physio={wp:.2f} (dev F1={best_f1:.3f})")

# ─── 5️⃣ APPLY ON TEST & REPORT ───────────────────────────────────────────
fused_te = wt*tp_te + wa*ap_te + wp*pp_te
pred_te  = (fused_te>0.5).astype(int)

print("\n[Fused Stress vs Non-Stress on MELD Test]")
print("Accuracy:", accuracy_score(y_te_bin, pred_te))
print("F1-Score:", f1_score(y_te_bin, pred_te))
print(classification_report(y_te_bin, pred_te, target_names=["non-stress","stress"]))



=== Physio-Only RF on WESAD Test ===
              precision    recall  f1-score   support

    baseline       0.97      0.97      0.97      2641
      stress       0.97      0.98      0.97      1494
   amusement       0.94      0.89      0.92       837
  meditation       0.95      0.97      0.96      1772

    accuracy                           0.96      6744
   macro avg       0.96      0.95      0.95      6744
weighted avg       0.96      0.96      0.96      6744


▶️ MELD splits: train=2610, dev=1109, test=2610

=== Text-only LR on MELD Test ===
              precision    recall  f1-score   support

           0       0.42      0.13      0.19      1256
           1       0.12      0.10      0.11       281
           2       0.00      0.02      0.01        50
           3       0.08      0.16      0.11       208
           4       0.15      0.18      0.17       402
           5       0.03      0.18      0.06        68
           6       0.12      0.18      0.14       345

    accur

In [7]:
from sklearn.ensemble     import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics      import classification_report, accuracy_score

# 1️⃣ Align physio → utterance counts:
#    (We reuse the same simple slicing you used for late fusion)
Xp_uf_tr  = Xp_tr[:N1]            # first N1 WESAD windows → train utterances
Xp_uf_dev = Xp_tr[N1:N1+N2]       # next N2 windows → dev utterances
Xp_uf_te  = Xp_te[:N3]            # first N3 windows → test utterances

# 2️⃣ Build early‐fusion feature matrices
#    Train on train+dev, evaluate on test:
X_early_tr  = np.hstack([Xp_uf_tr,  X_text_tr,  X_audio_tr])
X_early_dev = np.hstack([Xp_uf_dev, X_text_dev, X_audio_dev])
X_early_te  = np.hstack([Xp_uf_te,  X_text_te,  X_audio_te])

# Combine train+dev for final training
X_train_all = np.vstack([X_early_tr, X_early_dev])
y_train_all = np.hstack([y_train,     y_dev])

# 3️⃣ Optional: scale the fused features
scaler = StandardScaler().fit(X_train_all)
X_train_all = scaler.transform(X_train_all)
X_early_te  = scaler.transform(X_early_te)

# 4️⃣ Train a single classifier on the concatenated features
clf = RandomForestClassifier(
    n_estimators=200, class_weight='balanced', random_state=42
)
clf.fit(X_train_all, y_train_all)

# 5️⃣ Evaluate on test
y_pred = clf.predict(X_early_te)
print("=== Early‐Fusion RF on Test ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


=== Early‐Fusion RF on Test ===
Accuracy: 0.4793103448275862
              precision    recall  f1-score   support

           0       0.48      1.00      0.65      1256
           1       0.00      0.00      0.00       281
           2       0.00      0.00      0.00        50
           3       0.00      0.00      0.00       208
           4       0.00      0.00      0.00       402
           5       0.00      0.00      0.00        68
           6       0.00      0.00      0.00       345

    accuracy                           0.48      2610
   macro avg       0.07      0.14      0.09      2610
weighted avg       0.23      0.48      0.31      2610



In [8]:
from sklearn.ensemble      import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics       import classification_report, accuracy_score

# ─── assume these are already defined ──────────────────────────────────────
# Xp_tr, Xp_te, yp_tr, yp_te           # from your WESAD split
# X_text_tr, X_text_dev, X_text_te     # from MELD text features
# X_audio_tr, X_audio_dev, X_audio_te  # from MELD audio features
# y_train (train labels), y_dev (dev labels), y_test (test labels)
# N1 = len(y_train), N2 = len(y_dev), N3 = len(y_test)

# 1️⃣ ALIGN WESAD WINDOWS → utterance counts
#    map first N1 windows of Xp_tr to train, next N2 to dev, first N3 of Xp_te to test
Xp_uf_tr  = Xp_tr[        :N1 ]
Xp_uf_dev = Xp_tr[ N1 : N1+N2 ]
Xp_uf_te  = Xp_te[       :N3 ]

# sanity check shapes
assert Xp_uf_tr.shape[0]  == N1
assert Xp_uf_dev.shape[0] == N2
assert Xp_uf_te.shape[0]  == N3

# 2️⃣ BUILD EARLY‐FUSION MATRICES
X_early_tr  = np.hstack([Xp_uf_tr,  X_text_tr,  X_audio_tr])
X_early_dev = np.hstack([Xp_uf_dev, X_text_dev, X_audio_dev])
X_early_te  = np.hstack([Xp_uf_te,  X_text_te,  X_audio_te])

# combine train+dev for final training
X_train_all = np.vstack([X_early_tr, X_early_dev])
y_train_all = np.hstack([y_train,     y_dev])

# 3️⃣ SCALE FEATURES
scaler = StandardScaler().fit(X_train_all)
X_train_all = scaler.transform(X_train_all)
X_early_te  = scaler.transform(X_early_te)

# 4️⃣ TRAIN A SINGLE CLASSIFIER (7‐way)
clf = RandomForestClassifier(
    n_estimators=300,
    class_weight='balanced',
    random_state=42
)
clf.fit(X_train_all, y_train_all)

# 5️⃣ EVALUATE on MELD test
y_pred = clf.predict(X_early_te)
print("=== Early‐Fusion RF on MELD Test ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=[
    'neutral','surprise','fear','sadness','joy','disgust','anger'
]))


=== Early‐Fusion RF on MELD Test ===
Accuracy: 0.4793103448275862
              precision    recall  f1-score   support

     neutral       0.48      1.00      0.65      1256
    surprise       0.00      0.00      0.00       281
        fear       0.00      0.00      0.00        50
     sadness       0.00      0.00      0.00       208
         joy       0.00      0.00      0.00       402
     disgust       0.00      0.00      0.00        68
       anger       0.00      0.00      0.00       345

    accuracy                           0.48      2610
   macro avg       0.07      0.14      0.09      2610
weighted avg       0.23      0.48      0.31      2610



In [9]:
# ─── 0️⃣ INSTALL & IMPORTS ────────────────────────────────────────────────
!pip install --quiet scipy scikit-learn tqdm

import os, glob, pickle
import numpy as np
import pandas as pd
from collections import Counter
from scipy.signal            import butter, filtfilt, resample_poly, find_peaks
from sklearn.ensemble        import RandomForestClassifier, StackingClassifier
from sklearn.neural_network  import MLPClassifier
from sklearn.svm             import SVC
from sklearn.linear_model    import LogisticRegression
from sklearn.preprocessing   import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics         import classification_report, accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')


# ─── 1️⃣ BUILD & TRAIN WESAD PHYSIO MODEL ────────────────────────────────
def convert_units(sig, mod):
    if mod=='EDA':    return ((sig/65536.0)*3.0)/0.12
    if mod=='ECG':    return ((sig/65536.0)-0.5)*3000.0
    if mod=='RESP':   return ((sig/65536.0)-0.5)*100.0
    return sig

def lowpass(sig, cutoff, fs):
    nyq = fs*0.5
    if cutoff>=nyq or sig.size<10:
        return sig
    b,a = butter(4, cutoff/nyq, btype='low')
    try:
        return filtfilt(b,a,sig)
    except:
        return sig

def downsample(sig, frm, to, mod=None):
    if mod: sig = convert_units(sig, mod)
    if frm==to: return sig
    f = lowpass(sig, to*0.5, frm)
    return resample_poly(f, to, frm)

def basic_stats(x):
    x = x.flatten()
    return [x.mean(), x.std(), x.min(), x.max(),
            np.percentile(x,25), np.percentile(x,50), np.percentile(x,75),
            x.ptp(), np.mean(np.abs(np.diff(x)))]

def ecg_feats(x, fs):
    x = x.flatten()
    peaks,_ = find_peaks(x, distance=fs*0.5)
    rr = np.diff(peaks)/fs if len(peaks)>1 else np.array([])
    hr  = 60/rr.mean() if rr.size>0 else 0
    hrv = rr.std()    if rr.size>0 else 0
    return [hr, hrv] + basic_stats(x)

def build_physio(root="/kaggle/input/wesad-wearable-stress-affect-detection-dataset/WESAD"):
    fs_o, fs_e, fs_r = 700, 64, 4
    win, stride = fs_o*4, fs_o*2
    Xp, yp = [], []
    for pkl_fp in glob.glob(os.path.join(root, "S*", "S*.pkl")):
        with open(pkl_fp,'rb') as f:
            d = pickle.load(f, encoding='latin1')
        ecg, eda, resp = d['signal']['chest']['ECG'], d['signal']['chest']['EDA'], d['signal']['chest']['Resp']
        lbl700 = d['label']
        ecg_ds  = downsample(ecg, fs_o, fs_e, 'ECG')
        eda_ds  = downsample(eda, fs_o, fs_r, 'EDA')
        resp_ds = downsample(resp,fs_o, fs_r, 'RESP')
        for i in range(0, len(lbl700)-win+1, stride):
            lab,_ = Counter(lbl700[i:i+win]).most_common(1)[0]
            if lab not in (1,2,3,4): continue
            e0,e1 = int(i/fs_o*fs_e), int((i+win-1)/fs_o*fs_e)+1
            r0,r1 = int(i/fs_o*fs_r), int((i+win-1)/fs_o*fs_r)+1
            feats = ecg_feats(ecg_ds[e0:e1], fs_e) + basic_stats(eda_ds[r0:r1]) + basic_stats(resp_ds[r0:r1])
            Xp.append(feats); yp.append(lab-1)
    return np.array(Xp), np.array(yp)

X_physio, y_physio = build_physio()
Xp_tr, Xp_te, yp_tr, yp_te = train_test_split(
    X_physio, y_physio, test_size=0.3, stratify=y_physio, random_state=42
)

rf_physio = RandomForestClassifier(
    n_estimators=200, class_weight='balanced', random_state=42
).fit(Xp_tr, yp_tr)

print("\n[Physio-only RF]")
print(classification_report(yp_te, rf_physio.predict(Xp_te),
      target_names=['baseline','stress','amusement','meditation']))

physio_probs_tr = rf_physio.predict_proba(Xp_tr)
physio_probs_te = rf_physio.predict_proba(Xp_te)


# ─── 2️⃣ LOAD & PREPARE MELD TEXT/AUDIO FEATURES ─────────────────────────
AUDIOTEXT = "/kaggle/input/audiotext"
train_df = pd.read_csv(f"{AUDIOTEXT}/train_sent_emo.csv")
dev_df   = pd.read_csv(f"{AUDIOTEXT}/dev_sent_emo.csv")
test_df  = pd.read_csv(f"{AUDIOTEXT}/test_sent_emo.csv")
N1, N2, N3 = len(train_df), len(dev_df), len(test_df)
print(f"\n[MELD splits] train={N1} dev={N2} test={N3}")

# labels (7-way)
if 'label' in train_df.columns:
    y_train = train_df['label'].values
    y_dev   = dev_df['label'].values
    y_test  = test_df['label'].values
else:
    emo_map = {'neutral':0,'surprise':1,'fear':2,'sadness':3,'joy':4,'disgust':5,'anger':6}
    y_train = train_df['Emotion'].map(emo_map).astype(int).values
    y_dev   = dev_df['Emotion'].map(emo_map).astype(int).values
    y_test  = test_df['Emotion'].map(emo_map).astype(int).values

# load pickles
with open(f"{AUDIOTEXT}/text_emotion.pkl",'rb')  as f: text_data  = pickle.load(f)
with open(f"{AUDIOTEXT}/audio_emotion.pkl",'rb') as f: audio_data = pickle.load(f)

def extract_meld(data, total):
    feats=[]
    for split in data:
        for _,arr in split.items():
            feats.append(arr)
    feats = np.vstack(feats)
    if feats.shape[0]>total: feats=feats[:total]
    if feats.shape[0]<total:
        pad = total-feats.shape[0]
        feats = np.vstack([feats, np.zeros((pad,feats.shape[1]))])
    return feats

X_text_all  = extract_meld(text_data,  N1+N2+N3)
X_audio_all = extract_meld(audio_data, N1+N2+N3)

X_text_tr, X_text_dev, X_text_te   = np.split(X_text_all,  [N1, N1+N2])
X_audio_tr, X_audio_dev, X_audio_te= np.split(X_audio_all, [N1, N1+N2])

# scale
sc_t = StandardScaler().fit(np.vstack([X_text_tr,X_text_dev]))
X_text_tr  = sc_t.transform(X_text_tr)
X_text_dev = sc_t.transform(X_text_dev)
X_text_te  = sc_t.transform(X_text_te)

sc_a = StandardScaler().fit(np.vstack([X_audio_tr,X_audio_dev]))
X_audio_tr  = sc_a.transform(X_audio_tr)
X_audio_dev = sc_a.transform(X_audio_dev)
X_audio_te  = sc_a.transform(X_audio_te)


# ─── 3️⃣ IMPROVED UNIMODAL TRAINING & SELECTION ──────────────────────────
# TEXT
rf_text = RandomForestClassifier( n_estimators=300, class_weight='balanced', random_state=42 )
rf_text.fit(np.vstack([X_text_tr,X_text_dev]), np.hstack([y_train,y_dev]))
y_rf_text   = rf_text.predict(X_text_te)

mlp_text = MLPClassifier(hidden_layer_sizes=(256,128), max_iter=200, random_state=42)
mlp_text.fit(np.vstack([X_text_tr,X_text_dev]), np.hstack([y_train,y_dev]))
y_mlp_text = mlp_text.predict(X_text_te)

stack_text = StackingClassifier(
    estimators=[
        ('lr', LogisticRegression(max_iter=1000, class_weight='balanced')),
        ('rf', rf_text),
        ('mlp', mlp_text),
        ('svm', SVC(kernel='rbf', probability=True, class_weight='balanced'))
    ],
    final_estimator=LogisticRegression(class_weight='balanced', max_iter=1000),
    cv=3
)
stack_text.fit(np.vstack([X_text_tr,X_text_dev]), np.hstack([y_train,y_dev]))
y_stack_text = stack_text.predict(X_text_te)

print("\n=== Text RF ===");   print(classification_report(y_test, y_rf_text))
print("=== Text MLP ===");      print(classification_report(y_test, y_mlp_text))
print("=== Text Stacked ==="); print(classification_report(y_test, y_stack_text))

text_models = {
    'rf':    (rf_text,  f1_score(y_test, y_rf_text,  average='weighted')),
    'mlp':   (mlp_text, f1_score(y_test, y_mlp_text, average='weighted')),
    'stack': (stack_text, f1_score(y_test, y_stack_text, average='weighted'))
}
best_text_name, (best_text_model, _) = max(text_models.items(), key=lambda kv: kv[1][1])
print(f"\n→ Best TEXT: {best_text_name}")

# AUDIO
rf_aud = RandomForestClassifier( n_estimators=300, class_weight='balanced', random_state=42 )
rf_aud.fit(np.vstack([X_audio_tr,X_audio_dev]), np.hstack([y_train,y_dev]))
y_rf_aud   = rf_aud.predict(X_audio_te)

mlp_aud = MLPClassifier(hidden_layer_sizes=(256,128), max_iter=200, random_state=42)
mlp_aud.fit(np.vstack([X_audio_tr,X_audio_dev]), np.hstack([y_train,y_dev]))
y_mlp_aud = mlp_aud.predict(X_audio_te)

stack_aud = StackingClassifier(
    estimators=[
        ('lr', LogisticRegression(max_iter=1000, class_weight='balanced')),
        ('rf', rf_aud),
        ('mlp', mlp_aud),
        ('svm', SVC(kernel='rbf', probability=True, class_weight='balanced'))
    ],
    final_estimator=LogisticRegression(class_weight='balanced', max_iter=1000),
    cv=3
)
stack_aud.fit(np.vstack([X_audio_tr,X_audio_dev]), np.hstack([y_train,y_dev]))
y_stack_aud = stack_aud.predict(X_audio_te)

print("\n=== Audio RF ===");   print(classification_report(y_test, y_rf_aud))
print("=== Audio MLP ===");      print(classification_report(y_test, y_mlp_aud))
print("=== Audio Stacked ==="); print(classification_report(y_test, y_stack_aud))

audio_models = {
    'rf':    (rf_aud,  f1_score(y_test, y_rf_aud,  average='weighted')),
    'mlp':   (mlp_aud, f1_score(y_test, y_mlp_aud, average='weighted')),
    'stack': (stack_aud, f1_score(y_test, y_stack_aud, average='weighted'))
}
best_aud_name, (best_aud_model, _) = max(audio_models.items(), key=lambda kv: kv[1][1])
print(f"\n→ Best AUDIO: {best_aud_name}")


# ─── 4️⃣ LATE FUSION (Binary Stress vs Non-Stress) ────────────────────────
# melt text/audio to stress probs
stress_idxs = [2,3,5,6]
tp_dev = best_text_model.predict_proba(X_text_dev)[:, stress_idxs].sum(axis=1)
ap_dev = best_aud_model .predict_proba(X_audio_dev)[:, stress_idxs].sum(axis=1)
pp_dev = physio_probs_tr[N1:N1+N2, 1]    # physio class=1 is stress

tp_te  = best_text_model.predict_proba(X_text_te)[:, stress_idxs].sum(axis=1)
ap_te  = best_aud_model .predict_proba(X_audio_te)[:, stress_idxs].sum(axis=1)
pp_te  = physio_probs_te[:N3, 1]

y_dev_bin = np.isin(np.hstack([y_dev]), stress_idxs).astype(int)
y_te_bin  = np.isin(y_test, stress_idxs).astype(int)

# grid-search weights
best = (0,0,0,-1)
grid = np.linspace(0,1,21)
for wt in grid:
    for wa in grid:
        wp = 1 - wt - wa
        if wp<0 or wp>1: continue
        fused = wt*tp_dev + wa*ap_dev + wp*pp_dev
        preds = (fused>0.5).astype(int)
        f1 = f1_score(y_dev_bin, preds)
        if f1>best[3]:
            best = (wt,wa,wp,f1)

wt, wa, wp, _ = best
print(f"\n→ Best fusion weights (dev): text={wt:.2f}, audio={wa:.2f}, physio={wp:.2f}")

# apply on test
fused_te = wt*tp_te + wa*ap_te + wp*pp_te
pred_te  = (fused_te>0.5).astype(int)

print("\n[Fused Stress vs Non-Stress Test]")
print("Accuracy:", accuracy_score(y_te_bin, pred_te))
print("F1-Score:", f1_score(y_te_bin, pred_te))
print(classification_report(y_te_bin, pred_te, target_names=["non-stress","stress"]))



[Physio-only RF]
              precision    recall  f1-score   support

    baseline       0.97      0.97      0.97      2641
      stress       0.97      0.98      0.97      1494
   amusement       0.94      0.89      0.92       837
  meditation       0.95      0.97      0.96      1772

    accuracy                           0.96      6744
   macro avg       0.96      0.95      0.95      6744
weighted avg       0.96      0.96      0.96      6744


[MELD splits] train=2610 dev=1109 test=2610

=== Text RF ===
              precision    recall  f1-score   support

           0       0.48      0.93      0.63      1256
           1       0.19      0.01      0.03       281
           2       0.00      0.00      0.00        50
           3       0.06      0.00      0.01       208
           4       0.15      0.02      0.04       402
           5       0.00      0.00      0.00        68
           6       0.11      0.02      0.03       345

    accuracy                           0.46      26